In [1]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util, subprocess, sys
for package, module in [("torch-geometric", "torch_geometric"), ("duckdb", "duckdb")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import ast
import json
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import duckdb
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/content/drive/MyDrive/Threat-detection/gnn_edge_data"
GRAPH_DIR = os.path.join(DATA_DIR, "prepared_graphs")
MODEL_DIR = os.path.join(DATA_DIR, "models")
OUTPUT_DIR = os.path.join(DATA_DIR, "rag_input")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TEST_FILE = os.path.join(DATA_DIR, "test_edges_5min_enriched.parquet")
RAW_FILE = "/content/drive/MyDrive/Threat-detection/gnn_edge_data/combined_uwf_dataset.parquet"
RAG_METADATA_FILE = os.path.join(DATA_DIR, "test_rag_metadata_5min.parquet")

BINARY_PATH = os.path.join(MODEL_DIR, "binary_graphsage_best.pt")
MITRE_PATH = os.path.join(MODEL_DIR, "mitre_graphsage.pt")
TACTIC_PATH = os.path.join(MODEL_DIR, "tactic_graphsage.pt")
BINARY_THRESHOLD = 0.8609099984169006

print("Device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


In [2]:
class BinaryGraphSAGE(nn.Module):
    def __init__(self, hidden_size=64, dropout=0.2):
        super().__init__()
        self.conv1 = SAGEConv(6, hidden_size)
        self.conv2 = SAGEConv(hidden_size, hidden_size)
        self.edge_classifier = nn.Sequential(
            nn.Linear(hidden_size * 2 + 10, hidden_size),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size, 1)
        )
        self.dropout = dropout

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index))
        src, dst = data.target_edge_index
        return self.edge_classifier(
            torch.cat([x[src], x[dst], data.target_edge_attr], dim=1)
        ).squeeze(-1)


class EdgeLabelGraphSAGE(nn.Module):
    def __init__(self, num_classes, hidden_size=64, dropout=0.2):
        super().__init__()
        self.conv1 = SAGEConv(6, hidden_size)
        self.conv2 = SAGEConv(hidden_size, hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2 + 10, hidden_size),
            nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size, num_classes)
        )
        self.dropout = dropout

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index))
        src, dst = data.target_edge_index
        return self.classifier(
            torch.cat([x[src], x[dst], data.target_edge_attr], dim=1)
        )

In [3]:
test_graphs = torch.load(
    os.path.join(GRAPH_DIR, "test_graphs.pt"),
    map_location="cpu",
    weights_only=False
)

test_edges = (
    pd.read_parquet(TEST_FILE)
    .sort_values("window_start", kind="quicksort")
    .reset_index(drop=True)
)
test_edges["window_start"] = pd.to_datetime(test_edges["window_start"]).dt.tz_localize(None)

for graph in test_graphs:
    ids = graph.target_row_ids if hasattr(graph, "target_row_ids") else graph.target_row_index
    graph.rag_row_ids = ids.clone().long()

binary_model = BinaryGraphSAGE().to(device)
binary_model.load_state_dict(
    torch.load(BINARY_PATH, map_location=device, weights_only=False)
)

mitre_checkpoint = torch.load(MITRE_PATH, map_location=device, weights_only=False)
tactic_checkpoint = torch.load(TACTIC_PATH, map_location=device, weights_only=False)

mitre_model = EdgeLabelGraphSAGE(len(mitre_checkpoint["classes"])).to(device)
mitre_model.load_state_dict(mitre_checkpoint["model_state_dict"])

tactic_model = EdgeLabelGraphSAGE(len(tactic_checkpoint["classes"])).to(device)
tactic_model.load_state_dict(tactic_checkpoint["model_state_dict"])

binary_model.eval()
mitre_model.eval()
tactic_model.eval()

loader = DataLoader(test_graphs, batch_size=128, shuffle=False, num_workers=0)
print("Test edges:", len(test_edges))

Test edges: 232088


In [4]:
def destination_port_group(value):
    if value is None or pd.isna(value):
        return None
    try:
        port = int(float(value))
    except (TypeError, ValueError):
        return "unknown"

    named = {
        20: "FTP", 21: "FTP", 22: "SSH", 23: "Telnet", 25: "SMTP",
        53: "DNS", 80: "HTTP", 110: "POP3", 123: "NTP", 135: "RPC",
        139: "SMB", 143: "IMAP", 389: "LDAP", 443: "HTTPS", 445: "SMB",
        3306: "MySQL", 3389: "RDP", 5432: "PostgreSQL", 6379: "Redis",
        8080: "HTTP-alt"
    }
    if port in named:
        return named[port]
    if port <= 1023:
        return "well-known"
    if port <= 49151:
        return "registered"
    return "dynamic/private"


if os.path.exists(RAG_METADATA_FILE):
    rag_metadata = pd.read_parquet(RAG_METADATA_FILE)
    print("Loaded cached metadata:", RAG_METADATA_FILE)
else:
    if not os.path.exists(RAW_FILE):
        raise FileNotFoundError(
            "Update RAW_FILE to the location of combined_uwf_dataset.parquet: " + RAW_FILE
        )

    connection = duckdb.connect()
    connection.execute("""
        CREATE TEMP TABLE raw_test AS
        SELECT
            CAST(to_timestamp(floor(epoch(CAST(datetime AS TIMESTAMP)) / 300) * 300) AS TIMESTAMP) AS window_start,
            CAST(src_ip_zeek AS VARCHAR) AS src_ip_zeek,
            CAST(dest_ip_zeek AS VARCHAR) AS dest_ip_zeek,
            CAST(dest_port_zeek AS VARCHAR) AS destination_port,
            COALESCE(NULLIF(CAST(service AS VARCHAR), ''), 'unknown') AS zeek_service,
            COALESCE(NULLIF(CAST(conn_state AS VARCHAR), ''), 'unknown') AS conn_state
        FROM read_parquet(?)
        WHERE CAST(source_period AS VARCHAR) = '2025'
    """, [RAW_FILE])

    rag_metadata = connection.execute("""
        WITH base AS (
            SELECT
                window_start, src_ip_zeek, dest_ip_zeek,
                mode(destination_port) FILTER (
                    WHERE destination_port IS NOT NULL AND destination_port NOT IN ('nan', 'None')
                ) AS dominant_destination_port,
                mode(zeek_service) AS dominant_zeek_service
            FROM raw_test
            GROUP BY window_start, src_ip_zeek, dest_ip_zeek
        ),
        service_count AS (
            SELECT window_start, src_ip_zeek, dest_ip_zeek, zeek_service, count(*) AS n
            FROM raw_test
            GROUP BY window_start, src_ip_zeek, dest_ip_zeek, zeek_service
        ),
        service_json AS (
            SELECT window_start, src_ip_zeek, dest_ip_zeek,
                   json_group_object(zeek_service, n) AS service_counts
            FROM service_count
            GROUP BY window_start, src_ip_zeek, dest_ip_zeek
        ),
        state_count AS (
            SELECT window_start, src_ip_zeek, dest_ip_zeek, conn_state, count(*) AS n
            FROM raw_test
            GROUP BY window_start, src_ip_zeek, dest_ip_zeek, conn_state
        ),
        state_value AS (
            SELECT *, n * 1.0 / sum(n) OVER (
                PARTITION BY window_start, src_ip_zeek, dest_ip_zeek
            ) AS ratio
            FROM state_count
        ),
        state_json AS (
            SELECT window_start, src_ip_zeek, dest_ip_zeek,
                   json_group_object(conn_state, n) AS conn_state_counts,
                   json_group_object(conn_state, round(ratio, 6)) AS conn_state_ratios
            FROM state_value
            GROUP BY window_start, src_ip_zeek, dest_ip_zeek
        )
        SELECT base.*, service_json.service_counts,
               state_json.conn_state_counts, state_json.conn_state_ratios
        FROM base
        LEFT JOIN service_json USING (window_start, src_ip_zeek, dest_ip_zeek)
        LEFT JOIN state_json USING (window_start, src_ip_zeek, dest_ip_zeek)
    """).df()
    connection.close()

    rag_metadata["destination_port_group"] = rag_metadata[
        "dominant_destination_port"
    ].map(destination_port_group)
    rag_metadata.to_parquet(RAG_METADATA_FILE, index=False)
    print("Saved metadata cache:", RAG_METADATA_FILE)

rag_metadata["window_start"] = pd.to_datetime(rag_metadata["window_start"]).dt.tz_localize(None)
enrichment_columns = [
    "dominant_destination_port", "destination_port_group", "dominant_zeek_service",
    "service_counts", "conn_state_counts", "conn_state_ratios"
]
test_edges = test_edges.drop(columns=[c for c in enrichment_columns if c in test_edges.columns])
test_edges["_row_order"] = np.arange(len(test_edges))
test_edges = (
    test_edges.merge(
        rag_metadata,
        on=["window_start", "src_ip_zeek", "dest_ip_zeek"],
        how="left",
        validate="one_to_one",
        sort=False
    )
    .sort_values("_row_order", kind="stable")
    .drop(columns="_row_order")
    .reset_index(drop=True)
)
print("Metadata-enriched test edges:", len(test_edges))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved metadata cache: /content/drive/MyDrive/Threat-detection/gnn_edge_data/test_rag_metadata_5min.parquet
Metadata-enriched test edges: 232088


In [5]:
event_columns = [
    "window_start", "src_ip_zeek", "dest_ip_zeek", "connection_count",
    "total_duration", "maximum_duration", "total_orig_bytes", "total_resp_bytes",
    "total_orig_packets", "total_resp_packets", "tcp_count", "udp_count", "icmp_count",
    "dominant_destination_port", "destination_port_group", "dominant_zeek_service",
    "service_counts", "conn_state_counts", "conn_state_ratios"
]

all_records = []

with torch.inference_mode():
    for batch in loader:
        row_ids = batch.rag_row_ids.cpu().numpy().astype(int)
        batch = batch.to(device)

        binary_probabilities = torch.sigmoid(binary_model(batch)).cpu().numpy()
        attack_positions = np.flatnonzero(binary_probabilities >= BINARY_THRESHOLD)

        mitre_probabilities = tactic_probabilities = None
        if len(attack_positions) > 0:
            mitre_probabilities = torch.sigmoid(mitre_model(batch)).cpu().numpy()
            tactic_probabilities = torch.softmax(tactic_model(batch), dim=1).cpu().numpy()

        for position, row_id in enumerate(row_ids):
            attack_probability = float(binary_probabilities[position])
            is_attack = attack_probability >= BINARY_THRESHOLD
            source = test_edges.iloc[int(row_id)]

            record = {
                "row_id": int(row_id),
                "binary_label": "attack" if is_attack else "normal",
                "attack_probability": attack_probability,
                "binary_confidence": attack_probability if is_attack else 1.0 - attack_probability
            }
            for column in event_columns:
                record[column] = source.get(column)

            if is_attack:
                technique_ids = np.argsort(-mitre_probabilities[position])[:3]
                for rank, technique_id in enumerate(technique_ids, start=1):
                    record[f"mitre_top_{rank}"] = mitre_checkpoint["classes"][technique_id]
                    record[f"mitre_top_{rank}_confidence"] = float(
                        mitre_probabilities[position, technique_id]
                    )
                tactic_id = int(np.argmax(tactic_probabilities[position]))
                record["tactic"] = tactic_checkpoint["classes"][tactic_id]
                record["tactic_confidence"] = float(tactic_probabilities[position, tactic_id])
            else:
                for rank in range(1, 4):
                    record[f"mitre_top_{rank}"] = None
                    record[f"mitre_top_{rank}_confidence"] = None
                record["tactic"] = None
                record["tactic_confidence"] = None

            record["true_label_binary"] = int(source["binary_target"])
            record["true_label_technique"] = source.get("mitre_targets")
            record["true_label_tactic"] = source.get("tactic_targets")
            all_records.append(record)

all_predictions_df = pd.DataFrame(all_records)
print("Total events:", len(all_predictions_df))
print(all_predictions_df["binary_label"].value_counts())
display(all_predictions_df.head())

Total events: 232088
binary_label
normal    230614
attack      1474
Name: count, dtype: int64


,row_id,binary_label,attack_probability,binary_confidence,window_start,src_ip_zeek,dest_ip_zeek,connection_count,total_duration,maximum_duration,...,mitre_top_1_confidence,mitre_top_2,mitre_top_2_confidence,mitre_top_3,mitre_top_3_confidence,tactic,tactic_confidence,true_label_binary,true_label_technique,true_label_tactic
0,0,normal,0.276021,0.723979,2025-05-23,143.88.4.13,143.88.255.10,2,0.000024,0.000015,...,NaN,None,NaN,None,NaN,None,NaN,0,None,None
1,1,normal,0.238741,0.761259,2025-05-23,143.88.10.13,143.88.10.255,1,4.005710,4.005710,...,NaN,None,NaN,None,NaN,None,NaN,0,None,None
2,2,normal,0.246429,0.753571,2025-05-23,143.88.5.10,143.88.255.10,1,0.000014,0.000014,...,NaN,None,NaN,None,NaN,None,NaN,0,None,None
3,3,normal,0.276020,0.723980,2025-05-23,143.88.0.18,143.88.255.10,2,0.000061,0.000046,...,NaN,None,NaN,None,NaN,None,NaN,0,None,None
4,4,normal,0.276021,0.723979,2025-05-23,143.88.3.13,143.88.255.10,2,0.000028,0.000014,...,NaN,None,NaN,None,NaN,None,NaN,0,None,None


In [6]:
FLAT_PARQUET = os.path.join(OUTPUT_DIR, "combined_predictions_all_events.parquet")
all_predictions_df.to_parquet(FLAT_PARQUET, index=False)
print("Flat predictions:", FLAT_PARQUET)

Flat predictions: /content/drive/MyDrive/Threat-detection/gnn_edge_data/rag_input/combined_predictions_all_events.parquet


In [7]:
def clean_value(value):
    if value is None:
        return None
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [clean_value(item) for item in value]
    if isinstance(value, dict):
        return {str(k): clean_value(v) for k, v in value.items()}
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    return value


def parse_json_dict(value):
    if isinstance(value, dict):
        return clean_value(value)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return {}
    try:
        return clean_value(json.loads(str(value)))
    except (json.JSONDecodeError, TypeError):
        return {}


def parse_label_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [str(item) for item in value if str(item).lower() not in {"none", "nan", "unknown"}]
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass
    text = str(value).strip()
    if text.startswith("["):
        try:
            return parse_label_list(ast.literal_eval(text))
        except (ValueError, SyntaxError):
            pass
    return [] if text.lower() in {"", "none", "nan", "unknown"} else [text]


def create_gnn_json(row, include_ground_truth=False):
    mitre_candidates = []
    for rank in range(1, 4):
        technique_id = row.get(f"mitre_top_{rank}")
        if pd.notna(technique_id):
            mitre_candidates.append({
                "rank": rank,
                "technique_id": str(technique_id),
                "confidence": float(row[f"mitre_top_{rank}_confidence"])
            })

    tactic_prediction = None
    if pd.notna(row.get("tactic")):
        tactic_prediction = {
            "label": str(row["tactic"]),
            "confidence": float(row["tactic_confidence"]),
            "confidence_type": "softmax"
        }

    row_id = int(row["row_id"])
    result = {
        "event_id": f"uwf-test-{row_id}",
        "row_id": row_id,
        "event": {
            "window_start": clean_value(row.get("window_start")),
            "source_ip": clean_value(row.get("src_ip_zeek")),
            "destination_ip": clean_value(row.get("dest_ip_zeek")),
            "traffic_features": {
                "connection_count": clean_value(row.get("connection_count")),
                "total_duration": clean_value(row.get("total_duration")),
                "maximum_duration": clean_value(row.get("maximum_duration")),
                "total_orig_bytes": clean_value(row.get("total_orig_bytes")),
                "total_resp_bytes": clean_value(row.get("total_resp_bytes")),
                "total_orig_packets": clean_value(row.get("total_orig_packets")),
                "total_resp_packets": clean_value(row.get("total_resp_packets")),
                "tcp_count": clean_value(row.get("tcp_count")),
                "udp_count": clean_value(row.get("udp_count")),
                "icmp_count": clean_value(row.get("icmp_count")),
                "dominant_destination_port": clean_value(row.get("dominant_destination_port")),
                "destination_port_group": clean_value(row.get("destination_port_group")),
                "dominant_zeek_service": clean_value(row.get("dominant_zeek_service")),
                "service_counts": parse_json_dict(row.get("service_counts")),
                "conn_state_counts": parse_json_dict(row.get("conn_state_counts")),
                "conn_state_ratios": parse_json_dict(row.get("conn_state_ratios"))
            }
        },
        "gnn_detection": {
            "binary_prediction": {
                "label": str(row["binary_label"]),
                "attack_probability": float(row["attack_probability"]),
                "confidence": float(row["binary_confidence"]),
                "confidence_type": "sigmoid"
            },
            "selected_mitre_id": mitre_candidates[0]["technique_id"] if mitre_candidates else None,
            "mitre_confidence_type": "independent_sigmoid",
            "mitre_candidates": mitre_candidates,
            "tactic_prediction": tactic_prediction
        }
    }

    if include_ground_truth:
        result["ground_truth"] = {
            "label_binary": int(row["true_label_binary"]),
            "label_technique": parse_label_list(row.get("true_label_technique")),
            "label_tactic": parse_label_list(row.get("true_label_tactic"))
        }
    return result

In [8]:
production_path = os.path.join(OUTPUT_DIR, "gnn_attack_events_for_rag.jsonl")
evaluation_path = os.path.join(OUTPUT_DIR, "gnn_test_events_evaluation.jsonl")

with (
    open(production_path, "w", encoding="utf-8") as production_file,
    open(evaluation_path, "w", encoding="utf-8") as evaluation_file,
):
    for row in all_predictions_df.to_dict(orient="records"):
        evaluation_file.write(json.dumps(create_gnn_json(row, include_ground_truth=True)) + "\n")
        if row["binary_label"] == "attack":
            production_file.write(json.dumps(create_gnn_json(row, include_ground_truth=False)) + "\n")

print("Production attack JSONL:", production_path)
print("Evaluation all-event JSONL:", evaluation_path)
print("Production attack records:", int((all_predictions_df["binary_label"] == "attack").sum()))
print("Evaluation records:", len(all_predictions_df))

Production attack JSONL: /content/drive/MyDrive/Threat-detection/gnn_edge_data/rag_input/gnn_attack_events_for_rag.jsonl
Evaluation all-event JSONL: /content/drive/MyDrive/Threat-detection/gnn_edge_data/rag_input/gnn_test_events_evaluation.jsonl
Production attack records: 1474
Evaluation records: 232088


In [10]:
example_rows = (
    all_predictions_df[
    (all_predictions_df["binary_label"] == "attack")
    & (all_predictions_df["true_label_binary"] == 1)
]
    .head(10)
    .to_dict(orient="records")
)

example_jsons = [
    create_gnn_json(row, include_ground_truth=True)
    for row in example_rows
]

for number, example in enumerate(example_jsons, start=1):
    print(f"\n--- JSON example {number} ---")
    print(json.dumps(example, indent=2))


--- JSON example 1 ---
{
  "event_id": "uwf-test-135951",
  "row_id": 135951,
  "event": {
    "window_start": "2025-05-30T16:25:00",
    "source_ip": "143.88.3.14",
    "destination_ip": "143.88.3.11",
    "traffic_features": {
      "connection_count": 65536,
      "total_duration": 5.780602693557739,
      "maximum_duration": 0.003136873245239258,
      "total_orig_bytes": 0.0,
      "total_resp_bytes": 0.0,
      "total_orig_packets": 65535.0,
      "total_resp_packets": 62691.0,
      "tcp_count": 65536.0,
      "udp_count": 0.0,
      "icmp_count": 0.0,
      "dominant_destination_port": null,
      "destination_port_group": null,
      "dominant_zeek_service": null,
      "service_counts": {},
      "conn_state_counts": {},
      "conn_state_ratios": {}
    }
  },
  "gnn_detection": {
    "binary_prediction": {
      "label": "attack",
      "attack_probability": 0.9925090670585632,
      "confidence": 0.9925090670585632,
      "confidence_type": "sigmoid"
    },
    "selected_